# Pré-processamento — Base NPS Azul

Este notebook integra as fontes de `data/raw`, documenta o diagnóstico e produz uma matriz preparada para modelagem sem alterar a referência bruta.

Princípios adotados:

- preservar a base integrada original;
- não remover registros, outliers ou nulos automaticamente;
- investigar valores ausentes antes de qualquer tratamento;
- registrar decisões para discussão do grupo;
- ajustar imputação, codificação e escala somente no treino, evitando vazamento de dados.

## 1. Descoberta, integração e preservação da base bruta

A rotina descobre arquivos compatíveis em `data/raw`, ignora a distribuição populacional e compara cópias em formatos diferentes antes de concatenar. Ela usa merge `one_to_one`. Para a duplicidade aprovada de `RESPONDENT_ID`, consolida uma única linha somente se todas as demais colunas forem iguais e calcula a média de `TEMPO_VOO`; qualquer outra divergência continua bloqueando a integração. Após a validação da chave, `df_raw` permanece como referência e `df` é a cópia de trabalho.

## Conexão com a base de dados

As bases da Azul **não são versionadas** neste repositório, por compromisso de proteção de dados firmado entre o Inteli e o parceiro: a pasta `data/` e as extensões `.csv`, `.xlsx` e `.parquet` estão no `.gitignore`. Para executar este notebook:

1. **Arquivos brutos.** Copie os arquivos `.csv` recebidos da Azul para `data/raw/`, na raiz do repositório:
   - `PROJETO_INTELI.NPS_01.csv` a `PROJETO_INTELI.NPS_04.csv`;
   - `PROJETO_INTELI.PERFIL_CLIENTE_01.csv` e `PROJETO_INTELI.PERFIL_CLIENTE_02.csv`;
   - `PROJETO_INTELI.INFORMACAO_VIAGEM.csv`;
   - `PROJETO_INTELI.DISTRIBUICAO_PAX_NORMALIZADO.csv`.

   Deixe fora de `data/raw/` as cópias `.xlsx` das mesmas bases: o pré-processamento compara os dois formatos de uma mesma parte e interrompe a execução se eles divergirem.
2. **Base analítica.** Execute `notebooks/pre-processamento.ipynb`. Ele integra os arquivos de `data/raw/` e grava `data/processed/base_analitica.parquet`, a base que os notebooks de análise e de modelagem leem.
3. **Google Colab.** Siga a seção "No Google Colab" do `README.md`: envie a pasta do projeto ao Google Drive, com as bases em `data/raw/`, e execute antes de tudo uma célula que monta o Drive e entra na pasta `notebooks/` do projeto (`%cd`). Com isso, os caminhos descritos abaixo funcionam sem ajuste, como na execução local.

**Neste notebook:** lê `data/raw/` e grava `data/processed/base_analitica.parquet`. A primeira célula de código procura `data/raw/` a partir da raiz do repositório, de `notebooks/` e de `/content/` (Colab). Para usar outro local, acrescente o caminho à lista `possiveis_pastas`.

In [ ]:
!pip install -q pandas numpy scikit-learn openpyxl pyarrow python-calamine

import pandas as pd
import sys
import importlib
from pathlib import Path

# Inclui a pasta do módulo tanto no projeto local quanto no Colab.
for pasta_codigo in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))

import preprocessamento_nps
importlib.reload(preprocessamento_nps)

from preprocessamento_nps import integrar_bases

# Localiza a pasta que contém os arquivos brutos.
possiveis_pastas = [
    Path("/content/data/raw"),
    Path("/content/projeto/data/raw"),
    Path("data/raw"),
    Path("../data/raw"),
]

pasta_dados = next(
    (pasta for pasta in possiveis_pastas if pasta.exists()),
    None
)

if pasta_dados is None:
    raise FileNotFoundError(
        "Não encontrei data/raw. Execute: !find /content -type d -name raw"
    )

print("Usando dados em:", pasta_dados)
raiz_projeto = pasta_dados.parent.parent

df_integrado, relatorio_integracao = integrar_bases(pasta_dados)
df_raw = df_integrado.copy()
df = df_raw.copy()

display(relatorio_integracao)
df.shape

## 2. Integridade da chave `RESPONDENT_ID` e consolidação aprovada

Antes do merge, o ID `49088377` aparecia duas vezes. Verificamos que as linhas eram iguais em todas as demais colunas e divergiam apenas em `TEMPO_VOO` (340 e 1.084 minutos). Para preservar uma única observação por respondente sem privilegiar arbitrariamente um dos valores, decidimos consolidá-las pela média aritmética: `TEMPO_VOO = 712` minutos. A coluna `TEMPO_VOO_CONSOLIDADO` recebe 1 para registrar essa intervenção, enquanto os arquivos de origem em `data/raw` permanecem inalterados.

In [ ]:
print('RESPONDENT_ID únicos:', df['RESPONDENT_ID'].nunique())
print('Linhas com RESPONDENT_ID duplicado:', df['RESPONDENT_ID'].duplicated(keep=False).sum())

display(df.loc[
    df['RESPONDENT_ID'].duplicated(keep=False),
    ['RESPONDENT_ID', 'TEMPO_VOO']
])

## 3. Perfil de qualidade dos dados

Esta tabela lista tipo, quantidade e percentual de nulos e cardinalidade de cada coluna. Ela orienta a investigação sem aplicar `dropna()`, preenchimento ou transformação.

In [ ]:
perfil = pd.DataFrame({
    'coluna': df.columns,
    'tipo': df.dtypes.astype(str).values,
    'nulos': df.isna().sum().values,
    'percentual_nulos': (df.isna().mean() * 100).round(2).values,
    'valores_unicos': df.nunique(dropna=True).values,
})

perfil = perfil.sort_values(
    by=['percentual_nulos', 'valores_unicos'], ascending=[False, False]
).reset_index(drop=True)
perfil

## 4. Frequências das variáveis categóricas

Visualizamos as categorias mais frequentes para identificar valores constantes, grafias divergentes e colunas de alta cardinalidade. A consulta não altera os valores. `VOO_INTERNACIONAL`, por exemplo, é constante (`Domestic`) e poderá ser removida apenas da cópia processada após aprovação do grupo.

In [ ]:
categoricas = df.select_dtypes(include=['object', 'string']).columns

for coluna in categoricas:
    print(f'\n--- {coluna} ---')
    print('Valores únicos:', df[coluna].nunique(dropna=True))
    display(df[coluna].value_counts(dropna=False).head(10))

## 5. Investigação de entretenimento

As tabelas cruzadas verificam se os nulos são estruturais. A investigação mostrou que `TIPO_ENTRETENIMENTO` nulo não pode ser automaticamente convertido em “Não aplicável (conexão)”: 38,9% desses registros possuem uma resposta em `SUB_ENTRETENIMENTO1`. Portanto, os nulos são preservados.

In [ ]:
pd.crosstab(
    df['TIPO_ENTRETENIMENTO'].fillna('NULO'),
    df['VOO_TIPO'],
    margins=True
)

In [ ]:
pd.crosstab(
    df['TIPO_ENTRETENIMENTO'].fillna('NULO'),
    df['SUB_ENTRETENIMENTO1'].fillna('NULO'),
    normalize='index'
).mul(100).round(1)

## 6. Validação da pergunta condicional `SUB_ENTRETENIMENTO2`

`SUB_ENTRETENIMENTO2` pergunta qual foi o problema e, por isso, deve ser preenchida somente após `SUB_ENTRETENIMENTO1 = Sim`. A tabela abaixo confirmou a regra: as demais respostas têm 100% de nulos. Há 756 nulos (2,3%) entre os casos `Sim`; eles são preservados e registrados como pendência de qualidade.

In [ ]:
pd.crosstab(
    df['SUB_ENTRETENIMENTO1'].fillna('NULO'),
    df['SUB_ENTRETENIMENTO2'].fillna('NULO'),
    margins=True
)

## 7. Nulos de motivo e frequência de viagem

`SUB_FIL_MOTIVOVIAGEM` (0,61% nulos) e `SUB_FIL_FREQUENCIAAZUL` (0,81%) têm baixa ausência. A decisão é mantê-los nulos até que exista evidência para uma categoria explícita; não usamos moda nem apagamos registros.

In [ ]:
colunas_contexto = ['NPS_PRINCIPAL', 'VOO_TIPO', 'CANAL_COMPRA', 'SEGMENTO', 'TIER_VIAGEM']

for coluna in ['SUB_FIL_MOTIVOVIAGEM', 'SUB_FIL_FREQUENCIAAZUL']:
    print(f'\n--- Nulos em {coluna} ---')
    print('Quantidade:', df[coluna].isna().sum())
    print('Percentual:', round(df[coluna].isna().mean() * 100, 2), '%')
    for contexto in colunas_contexto:
        print(f'\n{contexto}:')
        display(df.loc[df[coluna].isna(), contexto].value_counts(dropna=False).to_frame('quantidade'))

## 8. Validação de `ANTECEDENCIA_CANCELAMENTO`

Verificamos se a ausência dessa variável representa voo sem cancelamento. O resultado esperado é nulo quando `CANCELAMENTO_VOO = False` e preenchido quando é `True`. Os nulos devem permanecer, pois preencher com zero confundiria ausência de cancelamento com cancelamento sem antecedência.

In [ ]:
pd.crosstab(
    df['CANCELAMENTO_VOO'].fillna('NULO'),
    df['ANTECEDENCIA_CANCELAMENTO'].isna(),
    margins=True
)

## 9. Conversão segura de data

`DATA_STD` é validada na leitura de cada arquivo, antes da concatenação. `format='mixed'` interpreta formatos individuais sem alterar dia, mês ou ano; `errors='coerce'` converte datas ausentes ou inválidas em `NaT` e a integração é interrompida se houver qualquer ocorrência. A coluna original é preservada e esta célula repete a verificação ao criar a coluna de trabalho.

In [ ]:
df['DATA_STD_CONVERTIDA'] = pd.to_datetime(
    df['DATA_STD'],
    format='mixed',
    errors='coerce'
)

datas_invalidas = df['DATA_STD_CONVERTIDA'].isna().sum()
if datas_invalidas:
    raise ValueError(f'DATA_STD_CONVERTIDA possui {datas_invalidas} data(s) inválida(s).')
print('Datas inválidas após conversão:', datas_invalidas)
df[['DATA_STD', 'DATA_STD_CONVERTIDA']].head()

## 10. Identificação de outliers

A regra IQR e os percentis são usados apenas para localizar valores extremos, sem excluir ou corrigir observações. A interpretação deve considerar o contexto: atrasos altos podem explicar detração, conexões podem ter longa duração, e alta frequência de viagem pode ser comportamento legítimo.

In [ ]:
variaveis_quantitativas = [
    'TEMPO_VOO',
    'ESTATISTICA_ATRASOSAIDA',
    'ATRASO_CHEGADA',
    'ANTECEDENCIA_CANCELAMENTO',
    'QTDE_VIAGENS_12M',
    'QTDE_VIAGENS_24M',
    'QTDE_VIAGENS_36M',
]

resumo_outliers = []

for coluna in variaveis_quantitativas:
    serie = pd.to_numeric(df[coluna], errors='coerce').dropna()
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    resumo_outliers.append({
        'variavel': coluna,
        'count': serie.count(),
        'media': serie.mean(),
        'mediana': serie.median(),
        'desvio_padrao': serie.std(),
        'min': serie.min(),
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'p95': serie.quantile(0.95),
        'p99': serie.quantile(0.99),
        'max': serie.max(),
        'outliers_iqr': ((serie < limite_inferior) | (serie > limite_superior)).sum(),
    })

resumo_outliers = pd.DataFrame(resumo_outliers).round(2)
resumo_outliers

## 11. `TEMPO_VOO` por tipo de voo

A comparação permite verificar se tempos altos são compatíveis com conexões. Os resultados observados mostram P99 de 1.405 minutos em conexões, contra 225 em voos diretos. Por isso, nenhum tempo de voo é removido; os extremos de direto e escala ficam como pendência de validação com a fonte dos dados.

In [ ]:
tempo_por_tipo = df.groupby('VOO_TIPO')['TEMPO_VOO'].agg(
    count='count',
    mediana='median',
    p95=lambda x: x.quantile(0.95),
    p99=lambda x: x.quantile(0.99),
    max='max'
).round(2)
tempo_por_tipo

## 12. Variáveis analíticas seguras

Criamos `MES_ANO` para análises temporais, `DETRATOR` para a modelagem binária e `CATEGORIA_NPS` para análises multiclasse (detrator, neutro e promotor). As colunas originais permanecem intactas.

In [ ]:
from preprocessamento_nps import preparar_base_analitica

df = preparar_base_analitica(df_raw)
print('TEMPO_VOO inválido (negativo ou zero), convertido em nulo:', df['TEMPO_VOO_INVALIDO'].sum())
display(df[['MES_ANO', 'NPS_PRINCIPAL', 'DETRATOR', 'CATEGORIA_NPS', 'TEMPO_VOO_INVALIDO']].head())

distribuicao_detrator = df['DETRATOR'].value_counts().rename_axis('DETRATOR').to_frame('quantidade')
distribuicao_detrator['percentual'] = (distribuicao_detrator['quantidade'] / len(df) * 100).round(2)
distribuicao_detrator

## 13. Persistência da base analítica em Parquet

Esta célula grava a cópia analítica em `data/processed/base_analitica.parquet`. Ela não altera os arquivos brutos. Nas próximas execuções, é possível carregar esse arquivo diretamente e seguir para a modelagem, sem repetir integração e diagnóstico.

In [ ]:
caminho_parquet = raiz_projeto / 'data' / 'processed' / 'base_analitica.parquet'
caminho_parquet.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(caminho_parquet, index=False, compression='zstd')

# Verificação: recarrega o arquivo salvo e confere a estrutura.
df_parquet = pd.read_parquet(caminho_parquet)
assert df_parquet.shape == df.shape
print(f'Base analítica salva em: {caminho_parquet}')
print(f'Formato validado: {df_parquet.shape[0]:,} linhas e {df_parquet.shape[1]} colunas')

## 14. Codificação, imputação e normalização para modelagem

A base analítica acima preserva os nulos estruturais. Nesta seção, eles recebem uma categoria explícita somente na matriz do modelo. As variáveis numéricas recebem mediana e indicador de ausência; em seguida, `RobustScaler` reduz a influência dos extremos reais preservados. As categóricas recebem `OneHotEncoder(handle_unknown='ignore')`. Tudo é ajustado exclusivamente em treino e aplicado ao teste, prevenindo vazamento. IDs, data textual, campos NPS e subperguntas são excluídos por identificarem registros ou por não estarem disponíveis no momento da predição.

In [ ]:
from preprocessamento_nps import criar_preprocessador_modelagem

preprocessador, dados_divididos, matrizes, colunas_modelo, metadados_divisao = criar_preprocessador_modelagem(df)
x_treino, x_teste, y_treino, y_teste = dados_divididos
x_treino_pronto, x_teste_pronto = matrizes
numericas, categoricas = colunas_modelo

print(f'Numéricas normalizadas: {len(numericas)} | Categóricas codificadas: {len(categoricas)}')
print('Matriz de treino:', x_treino_pronto.shape)
print('Matriz de teste:', x_teste_pronto.shape)
print('Divisão temporal:', metadados_divisao)
assert len(x_treino) == len(y_treino) and len(x_teste) == len(y_teste)